### Deconvolution approach to filter the noise of the data, then measuring the ARI

Imports

In [6]:
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
import numpy as np
from hdbscan import HDBSCAN
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from fast_hdbscan.boruvka import sample_weight_core_distance



Dataset

In [ ]:
centers = [[-4.0, 0.0], [-4.0, 3.0], [4.0, 0.0], [4.0, 3.0]]
clusters_std = [0.5, 3.0, 0.5, 3.0]

n_per_comp = [100, 100, 100, 100]

X, raw_labels = make_blobs(
    n_samples = n_per_comp,
    centers = centers,
    cluster_std = clusters_std,
    random_state = 42
)

true_labels = raw_labels // 2

plt.scatter(X[:,0], X[:,1], c=true_labels, cmap='viridis')
plt.title("Ground-Truth Blobs")
plt.show()

Noisy dataset

In [ ]:
std = 1.0
X_obs = X + np.random.normal(scale=std, size=X.shape)

plt.scatter(X_obs[:,0], X_obs[:,1], c=true_labels, cmap="viridis")
plt.title("Noisy Blobs")
plt.show()

Making the grid to evaluate the frequencies

In [ ]:
T, N = 50, 2**12                   
t = np.linspace(-T, T, N)          
x_grid = np.linspace(             
    X_obs.min(),                  
    X_obs.max(),                  
    N                              
)

Making empirical characteristic function of noisy data 

In [ ]:
phi_X = np.mean(
    np.exp(1j * np.outer(X_obs.flatten(), t)),
    axis=0
)

phi_U = np.exp(-0.5 * (std**2) * t**2)

Kernel window in frequency domain

In [ ]:
h = 0

phi_K = (1 - (t*h)**2)**3          
phi_K[np.abs(t*h) > 1] = 0.0 

Form deconvolution multiplier and inverter

In [ ]:
G = phi_X * phi_K / phi_U
fhat = np.fft.ifftshift(
           np.fft.ifft(
             np.fft.ifftshift(G)
           )
       ).real / (2*np.pi)

Interpolator

In [ ]:
from scipy.interpolate import interp1d
fhat_interp = interp1d(
    x_grid,      
    fhat,        
    fill_value="extrapolate"
)

Computer core distances

In [ ]:
n = 0
k = 5

X_obs_fixed = X + np.random.normal(scale=std, size=X.shape)

d_obs = np.linalg.norm(
    X_obs_fixed[:, None, :] - X_obs_fixed[None, :, :],
    axis=2
)

core_dist_unc = np.zeros(n)        
for i in range(n):
    dists   = np.linalg.norm(         
        X_obs[i] - X_obs,
        axis=1
    )
    weights = fhat_interp(dists)      
    idx     = np.argsort(dists)       
    cum     = 0.0                     
    for j in idx:                     
        cum += weights[j]             
        if cum >= k:                  
            core_dist_unc[i] = dists[j]  
            break
    else:
        core_dist_unc[i] = dists[idx[-1]]

cd_i = core_dist_unc[:, None]   
cd_j = core_dist_unc[None, :]

assert cd_i.shape == (n, 1) and cd_j.shape == (1, n)

mutual = np.maximum(np.maximum(cd_i, cd_j), d_obs)

clusterer = HDBSCAN(
    metric='precomputed',     
    min_samples=k,
    min_cluster_size=k
)

pred_labels = clusterer.fit_predict(mutual)

ari = adjusted_rand_score(labels_true=true_labels, labels_pred=pred_labels)
print("ARI (uncertainty-aware), variant 4:", ari)